# unbind-tuple-unpack composite — cx18: tuple-unpack (O, D) from stacked rays then evaluate at a per-ray t vector

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `unbind-tuple-unpack`, `ray-parametric-form`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "unbind-tuple-unpack"
DD_ATOM_IDS = ["unbind-tuple-unpack", "ray-parametric-form"]
DD_SUBTOPICS = ["PyTorch: Unbind tuple-unpack", "Geometry: Ray parametric form"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Where cx13 used a scalar `t`, the real ARENA intersection code evaluates rays at a PER-RAY parameter — typically `s: (NR,)` returned by the batched linalg.solve. The composition is the same shape (unbind → parametric eval) but with broadcasting subtlety.

Given `rays: (NR, 2, 3)` and `t_vec: (NR,)`, the parametric form `P_i = O_i + t_vec_i * D_i` requires the scalar `t_vec_i` to broadcast against the 3-vector `D_i`. Standard broadcasting DOES NOT do this directly: `t_vec: (NR,) * D: (NR, 3)` aligns the trailing axes WRONG — `(NR,)` tries to align against the 3-axis. The fix is to add a trailing 1-axis: `t_vec[:, None]` (or `rearrange(t_vec, 'nr -> nr 1')`), giving `(NR, 1)` which broadcasts against `(NR, 3)` correctly.

Atom focus: `unbind-tuple-unpack` is the named-binding step; `ray-parametric-form` is the broadcasting-aware evaluation. Both load-bearing.

### Composite Exercise — tuple-unpack (O, D) from stacked rays then evaluate at a per-ray t vector

**Atoms exercised together**: `unbind-tuple-unpack`, `ray-parametric-form`

Implement `cx18_ray_at_per_ray_t(rays, t_vec)` where:

- `rays: (NR, 2, 3)` — axis 1 = [origin, direction]
- `t_vec: (NR,)` — a per-ray scalar parameter

Return `P: (NR, 3)` with `P_i = O_i + t_vec_i * D_i`.

1. **Tuple-unpack via unbind**: `O, D = t.unbind(rays, dim=1)` — the arity-2 unpack asserts that axis 1 has size 2.
2. **Broadcasting fix-up**: `t_vec[:, None]` to reshape `(NR,)` → `(NR, 1)` so it broadcasts against `D: (NR, 3)` along the correct axis. (Plain `t_vec * D` is a bug — wrong axis alignment.)
3. **Parametric form**: `O + t_vec[:, None] * D`.

Return shape `(NR, 3)`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx18_ray_at_per_ray_t(rays, t_vec):
    raise NotImplementedError

def _test_cx18():
    # Case A: hand-built rays at origin, all directions = +x; per-ray t = [0, 1, 2, 3].
    # Expected: P_i = (i, 0, 0).
    NR = 4
    O = t.zeros(NR, 3)
    D = t.zeros(NR, 3); D[:, 0] = 1.0
    rays = t.stack([O, D], dim=1)
    t_vec = t.tensor([0.0, 1.0, 2.0, 3.0])
    out = cx18_ray_at_per_ray_t(rays, t_vec)
    assert tuple(out.shape) == (NR, 3), f'expected (NR,3), got {tuple(out.shape)}'
    expected = t.tensor([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [2.0, 0.0, 0.0], [3.0, 0.0, 0.0]])
    assert t.allclose(out, expected), f'got {out}\nexpected {expected}'

    # Case B: random rays + random t_vec — cross-check against an explicit per-ray loop.
    t.manual_seed(11)
    rays2 = t.randn(7, 2, 3)
    tv2 = t.randn(7)
    out2 = cx18_ray_at_per_ray_t(rays2, tv2)
    ref2 = t.stack([rays2[i, 0] + tv2[i] * rays2[i, 1] for i in range(7)])
    assert t.allclose(out2, ref2, atol=1e-5), (
        'broadcast mismatch — did you forget t_vec[:, None]? '
        'Plain t_vec * D aligns t_vec against the 3-axis (wrong).'
    )

    # Case C: t_vec of all zeros — must reduce to origins exactly.
    out0 = cx18_ray_at_per_ray_t(rays2, t.zeros(7))
    assert t.allclose(out0, rays2[:, 0, :])

    # Case D: t_vec of all ones — must reduce to O + D.
    out1 = cx18_ray_at_per_ray_t(rays2, t.ones(7))
    ref1 = rays2[:, 0, :] + rays2[:, 1, :]
    assert t.allclose(out1, ref1)
    _dd_passed.add('cx18')

_test_cx18()

<details><summary>Show solution — cx18</summary>

```python
def cx18_ray_at_per_ray_t(rays, t_vec):
    # Atom A (unbind-tuple-unpack): named bindings for O and D. The arity-2 LHS asserts
    # at runtime that axis 1 has size 2.
    O, D = t.unbind(rays, dim=1)
    # Atom B (ray-parametric-form): need t_vec[:, None] so (NR, 1) broadcasts against
    # D: (NR, 3) on the correct axis. Plain t_vec * D would align (NR,) against the
    # 3-axis — wrong.
    return O + t_vec[:, None] * D
```

The `[:, None]` (or equivalently `.unsqueeze(-1)` / `rearrange(t_vec, 'nr -> nr 1')`) is the load-bearing broadcasting fix. Without it, PyTorch tries to align `(NR,)` against the trailing axis of `D: (NR, 3)` — i.e. the 3-axis — and you get either a shape error (if NR != 3) or, worse, *no* error and a silently-wrong result (if NR happens to equal 3). Always reshape your per-row scalar to `(NR, 1)` before broadcasting against a `(NR, D)` tensor. This is the same pattern as keepdim=True in cx27 but in the OPPOSITE direction (re-inserting an axis instead of preserving one).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx18'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx18',
        'subtopics': ["PyTorch: Unbind tuple-unpack", "Geometry: Ray parametric form"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()